In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/my_data/marketing_campaign.csv",
    header=True,
    inferSchema=True,
    sep="\t"  # marketing_campaign uses tab separator
)

print(f"Rows: {df.count()}")
print(f"Columns: {len(df.columns)}")
df.show(3)



Rows: 2240
Columns: 29
+----+----------+----------+--------------+------+-------+--------+-----------+-------+--------+---------+---------------+---------------+----------------+------------+-----------------+---------------+-------------------+-----------------+-----------------+------------+------------+------------+------------+------------+--------+-------------+---------+--------+
|  ID|Year_Birth| Education|Marital_Status|Income|Kidhome|Teenhome|Dt_Customer|Recency|MntWines|MntFruits|MntMeatProducts|MntFishProducts|MntSweetProducts|MntGoldProds|NumDealsPurchases|NumWebPurchases|NumCatalogPurchases|NumStorePurchases|NumWebVisitsMonth|AcceptedCmp3|AcceptedCmp4|AcceptedCmp5|AcceptedCmp1|AcceptedCmp2|Complain|Z_CostContact|Z_Revenue|Response|
+----+----------+----------+--------------+------+-------+--------+-----------+-------+--------+---------+---------------+---------------+----------------+------------+-----------------+---------------+-------------------+-----------------+-----

In [0]:
# Check column names and data types
df.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Year_Birth: integer (nullable = true)
 |-- Education: string (nullable = true)
 |-- Marital_Status: string (nullable = true)
 |-- Income: integer (nullable = true)
 |-- Kidhome: integer (nullable = true)
 |-- Teenhome: integer (nullable = true)
 |-- Dt_Customer: date (nullable = true)
 |-- Recency: integer (nullable = true)
 |-- MntWines: integer (nullable = true)
 |-- MntFruits: integer (nullable = true)
 |-- MntMeatProducts: integer (nullable = true)
 |-- MntFishProducts: integer (nullable = true)
 |-- MntSweetProducts: integer (nullable = true)
 |-- MntGoldProds: integer (nullable = true)
 |-- NumDealsPurchases: integer (nullable = true)
 |-- NumWebPurchases: integer (nullable = true)
 |-- NumCatalogPurchases: integer (nullable = true)
 |-- NumStorePurchases: integer (nullable = true)
 |-- NumWebVisitsMonth: integer (nullable = true)
 |-- AcceptedCmp3: integer (nullable = true)
 |-- AcceptedCmp4: integer (nullable = true)
 |-- AcceptedCmp

In [0]:
from pyspark.sql.functions import col, mean

# Check nulls
null_count = df.filter(col("Income").isNull()).count()
print(f"Null incomes: {null_count}")

# Fill nulls with mean
mean_income = df.select(mean("Income")).collect()[0][0]
df = df.fillna({"Income": mean_income})

# Confirm fixed
print(f"Nulls after fix: {df.filter(col('Income').isNull()).count()}")

Null incomes: 24
Nulls after fix: 0


In [0]:
from pyspark.sql.functions import when

# Add Age column
df = df.withColumn("Age", 2024 - col("Year_Birth"))

# Add Income Category
df = df.withColumn("Income_Category",
    when(col("Income") < 30000, "Low")
    .when(col("Income") < 60000, "Medium")
    .when(col("Income") < 90000, "High")
    .otherwise("Very High")
)

df.select("Year_Birth", "Age", "Income", "Income_Category").show(5)

+----------+---+------+---------------+
|Year_Birth|Age|Income|Income_Category|
+----------+---+------+---------------+
|      1957| 67| 58138|         Medium|
|      1954| 70| 46344|         Medium|
|      1965| 59| 71613|           High|
|      1984| 40| 26646|            Low|
|      1981| 43| 58293|         Medium|
+----------+---+------+---------------+
only showing top 5 rows


In [0]:
# Count by income category
df.groupBy("Income_Category").count().orderBy("count", ascending=False).show()

# Overall stats
df.select("Income", "Age").describe().show()

+---------------+-----+
|Income_Category|count|
+---------------+-----+
|         Medium| 1028|
|           High|  788|
|            Low|  370|
|      Very High|   54|
+---------------+-----+

+-------+------------------+------------------+
|summary|            Income|               Age|
+-------+------------------+------------------+
|  count|              2240|              2240|
|   mean|52247.248660714286| 55.19419642857143|
| stddev|25037.797168405727|11.984069456885834|
|    min|              1730|                28|
|    max|            666666|               131|
+-------+------------------+------------------+



In [0]:
from pyspark.sql.functions import col

# Add total spending column
df = df.withColumn("Total_Spending",
    col("MntWines") + col("MntFruits") + col("MntMeatProducts") +
    col("MntFishProducts") + col("MntSweetProducts") + col("MntGoldProds")
)

# Filter high value customers
high_value = df.filter(
    (col("Income") > 50000) & (col("Total_Spending") > 500)
)

print(f"High value customers: {high_value.count()}")
high_value.select("ID", "Income", "Total_Spending", "Income_Category").show(5)

High value customers: 940
+----+------+--------------+---------------+
|  ID|Income|Total_Spending|Income_Category|
+----+------+--------------+---------------+
|5524| 58138|          1617|         Medium|
|4141| 71613|           776|           High|
|7446| 62513|           716|           High|
| 965| 55635|           590|         Medium|
|2125| 63033|          1102|           High|
+----+------+--------------+---------------+
only showing top 5 rows


In [0]:
# Register as temp view to run SQL directly
df.createOrReplaceTempView("customers")

result = spark.sql("""
    SELECT
        Education,
        COUNT(*) AS customer_count,
        ROUND(AVG(Income), 2) AS avg_income,
        SUM(Total_Spending) AS total_spending,
        ROUND(AVG(Total_Spending), 2) AS avg_spending
    FROM customers
    GROUP BY Education
    ORDER BY total_spending DESC
""")

result.show()

+----------+--------------+----------+--------------+------------+
| Education|customer_count|avg_income|total_spending|avg_spending|
+----------+--------------+----------+--------------+------------+
|Graduation|          1127|  52715.75|        698626|       619.9|
|       PhD|           486|  56105.21|        326791|      672.41|
|    Master|           370|  52908.47|        226359|      611.78|
|  2n Cycle|           203|  47701.37|        100795|      496.53|
|     Basic|            54|  20306.26|          4417|        81.8|
+----------+--------------+----------+--------------+------------+

